# Appendix

This notebook contains additional code used to generate some ancillary inputs for the demos in the other notebooks.

## Compute $\overline{\mathrm{SIF}_j}$ and $\overline{\Delta \mathrm{SIF}_j}$ for a Region of Interest

To derive the Z-scores and RCI for the SIF data used in the first notebook, we need to determine mean SIF over the entire time range of our source data for each j-th 8-day window. For completeness, we will download the entire GOSIF dataset from 2001-2020 and compute the mean value per grid cell over the region of interest from the notebook, saving the result in a CSV. The download step will take about 20 minutes.

**Note:** The primary complexity in the second cell is due to the fact that Mohammadi and Wang compute z-scores of SIF anomaly _per grid cell_ rather than spatially averaging the data over the region of interest first and then computing the mean and standard devation of the SIF increment. To reiterate what was mentioned in the first notebook, the z-score involves dividing by the standard deviation of SIF incremement which is not commutative:
$$
Z(j, y) = \frac{\Delta \mathrm{SIF}_{j, y} - \overline{\Delta \mathrm{SIF}_j}}{\sigma_{\Delta \mathrm{SIF}_j}}
$$

In other words, the value for $\sigma_{\Delta \mathrm{SIF}_j}$ when computed per grid cell and then spatially averaged will differ from the value that would be obtained by spatially averaging first and then computing $\sigma_{\Delta \mathrm{SIF}_j}$. It's not strictly necessary to do this for $\overline{\mathrm{SIF}_j}$, the value in the third band (we could spatially average the data now), but we'll leave it gridded for now since there's no harm in doing so.

In [ ]:
import os

from download import download_unpack_gosif
from tqdm.notebook import tqdm

start_year = 2001
end_year = 2020

dates: list[tuple[int, int]] = []
for year in range(start_year, end_year + 1):
    for doy in range(1, 365, 8):
        dates.append((year, doy))

output_dir = "data/gosif"
os.makedirs(output_dir, exist_ok=True)

gosif_geotiffs: list[str] = []
for date_tuple in tqdm(dates, desc="Downloading granules"):
    fname = download_unpack_gosif(
        date_tuple[0],
        day=date_tuple[1],
        output_dir=output_dir,
        verbose=False
    )
    if fname:
        gosif_geotiffs.append(fname)

In [ ]:
import contextlib
import os
import warnings
from glob import glob

import numpy as np
import rasterio
from rasterio.windows import from_bounds
from rasterio.windows import transform as window_transform

"""Instructions
If you want to study your own region using this notebook, you need to generate new climatology files for the 2001-2020 period. While you _can_ do this
by putting in a small region of interest like you see below, if you are running this notebook yourself and have already downloaded 20 years worth of
GOSIF data in the above cell, you can also generate global climatology files in this cell like so:
region_name = ""
west, south, east, north = -180.0, -90.0, 180.0, 90.0

Then any subsequent region you study can use the global files as reference. The only reason this was not done for you is because it would have added
a large amount of data to the git repository for this course.
If you still wish to create a set of climatology files for a small region, set these variables in this cell:
1. region_name (str): A name for your region of interest. This will be used in the name of the directory that outputs get stored.
2. west, south, east, north (four floats): The corner coordinates of your region of interest.
"""

region_name = "" # "south_africa"
# Northern Great Plains region of interest
# 45.00°-50.00°N, 106.00°-111.00°W
west, south, east, north = -111.0, 45.0, -106.0, 50.0
# Free State South Africa region of interest
# 28.00°-29.00°S, 27.00°-28.50°E
# west, south, east, north = 27, -29.0, 28.5, -28.0

if region_name == "":
    output_dir = "inputs/sif_increments"
else:
    output_dir = f"inputs/sif_increments_{region_name}"
os.makedirs(output_dir, exist_ok=True)

input_dir = "data/gosif"
gosif_geotiffs = sorted(glob(f"{input_dir}/GOSIF_???????.tif"))

if "start_year" not in locals():
    start_year = 2001
if "end_year" not in locals():
    end_year = 2020

# The threshold and scale factor parameters come from the documentation:
# https://data.globalecology.unh.edu/data/GOSIF_v2/Fair_Data_Use_Policy_and_Readme_GOSIF_v2.pdf
# 32767 = water bodies, 32766 = ice/snow
gosif_data_thresh = 32765
# This value tells our code the conversion between pixel values in the GeoTIFF images to units of W/m^2/sr/μm
gosif_scale_factor = 0.0001

N_WINDOWS = 46  # GOSIF 8-day windows per year: DOY 1, 9 ... 361
ddof = 1        # 1 = sample std, 0 = population

# Georeferencing of the ROI window (identical across all GOSIF files)
with rasterio.open(gosif_geotiffs[0]) as src:
    read_window = from_bounds(west, south, east, north, src.transform)
    read_window = read_window.round_offsets().round_lengths()
    win_transform = window_transform(read_window, src.transform)
    crs = src.crs
height = int(read_window.height)
width = int(read_window.width)


def read_roi(path: str) -> np.ndarray:
    """Read the ROI, mask non-data (water/ice/fill), scale to physical units."""
    with rasterio.open(path) as src:
        arr = src.read(1, window=read_window).astype("float64")
    arr[arr > gosif_data_thresh] = np.nan
    return arr * gosif_scale_factor

def parse_year_doy(path: str) -> tuple[int, int]:
    """GOSIF_YYYYDDD.tif -> (year, doy)."""
    name = os.path.splitext(os.path.basename(path))[0]
    token = name.split("_")[1]
    return int(token[:4]), int(token[4:7])

def window_index(doy: int) -> int:
    """Map a GOSIF day-of-year to a window index j"""
    return (doy - 1) // 8

def window_ordinal(year: int, j: int) -> int:
    """Global ordinal of an 8-day window. Consecutive windows differ by 1,
    including across year boundaries (year*46 + 45  ->  (year+1)*46 + 0)."""
    return year * N_WINDOWS + j

@contextlib.contextmanager
def _suppress_runtime_warnings():
    """nanmean/nanstd warn on all-NaN or <=1-sample pixels; those become NaN,
    which is the intended 'no climatology here' result."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        yield


# Step 1: accumulate increment grids per jth 8-day window
increments_by_j: dict[int, list[np.ndarray]] = {j: [] for j in range(N_WINDOWS)}
values_by_j: dict[int, list[np.ndarray]] = {j: [] for j in range(N_WINDOWS)}
prev_grid: np.ndarray | None = None
prev_ord: int | None = None

for path in gosif_geotiffs:
    year, doy = parse_year_doy(path)
    j = window_index(doy)
    ordi = window_ordinal(year, j)

    # read 'start_year - 1' as lead-in (to seed the j=0 increment) but
    # store nothing from it; ignore everything outside [start_year-1, end_year]
    if year < start_year - 1 or year > end_year:
        prev_grid, prev_ord = None, None
        continue

    grid = read_roi(path)
    if start_year <= year <= end_year:
        values_by_j[j].append(grid)

        # store only a genuinely consecutive increment assigned to the study period
        if prev_grid is not None and prev_ord == ordi - 1:
            increments_by_j[j].append(grid - prev_grid)

    prev_grid, prev_ord = grid, ordi

# Step 2: per-pixel mean/std across years, write one GeoTIFF per j
profile = {
    "driver": "GTiff",
    "height": height,
    "width": width,
    "count": 4,
    "dtype": "float32",
    "crs": crs,
    "transform": win_transform,
    "nodata": np.nan,
    "compress": "deflate",
}

for j in range(N_WINDOWS):
    increment_list = increments_by_j[j]
    value_list = values_by_j[j]
    if not value_list or not increment_list:
        print(f"window j={j:02d} has no data, skipped")
        continue

    istack = np.stack(increment_list, axis=0)
    vstack = np.stack(value_list, axis=0)

    # per-pixel climatology; water/ice gaps ignored independently
    with np.errstate(invalid="ignore", divide="ignore"), _suppress_runtime_warnings():
        mean_incr = np.nanmean(istack, axis=0)
        std_incr = np.nanstd(istack, axis=0, ddof=ddof)
        mean_sif = np.nanmean(vstack, axis=0)
        std_sif = np.nanstd(vstack, axis=0, ddof=ddof)

    doy_j = j * 8 + 1
    out_path = os.path.join(output_dir, f"GOSIF_dSIF_clim_{doy_j:03d}.tif")
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(mean_incr.astype("float32"), 1)
        dst.write(std_incr.astype("float32"), 2)
        dst.write(mean_sif.astype("float32"), 3)
        dst.write(std_sif.astype("float32"), 4)
        dst.set_band_description(1, "mean dSIF")
        dst.set_band_description(2, "std dSIF")
        dst.set_band_description(3, "mean SIF")
        dst.set_band_description(4, "std SIF")
        dst.update_tags(
            window_index=str(j),
            doy=str(doy_j),
            start_year=str(start_year),
            end_year=str(end_year),
            ddof=str(ddof),
            units="W/m^2/sr/um",
        )
    print(f"window j={j:02d} (DOY {doy_j:03d}): wrote {out_path}")

## Virtualize Root Zone Soil Moisture Data from the Soil Moisture Active Passive (SMAP) Mission

To compute a time series of the Soil Water Deficit Index (SWDI) in the first notebook, we need to access the SMAP L4 Geophysical (GPH) data provided in the SPL4SMGP collection, doi: https://doi.org/10.5067/T5RUATAQREF8. This collection is derived from a combination of sources:
* SMAP L1C radiometer measurements
* Goddard Earth Observing System Forward Processing (GEOS-FP) surface meteorology model data
* NOAA Climate Prediction Center Unified (CPCU) precipitation data
* NASA  Integrated Multi-satellitE Retrievals for the Global Precipitation Measurement mission (IMERG) precipitation data.

The data are processed through the GEOS catchment land surface model to provide estimates of root zone soil moisture that we can use in our flash drought analysis. As the individual GPH granules are quite large (~150MB per 3-hour global file), it is efficient to "virtualize" the data before accessing it. If you wish to read more about virtual datasets, NASA's Physical Oceanography Distributed Active Archive Center has a writeup here: https://podaac.github.io/tutorials/quarto_text/UsingVirtualDatasets.html

In short, we want to **catalog each of the SPL4SMGP files over the 2017 growing season and store this catalog as a Kerchunk virtual zarr file.** This catalog does not download the data directly, but it can be opened later and treated as though it were a large Xarray dataset covering the entire time range we cataloged. The data are downloaded just-in-time and only for the specific variable(s) and spatial region of interest requested, so this represents a considerable improvement in data access efficiency. The cataloging process, or virtualization, is somewhat slow however so we will do it beforehand and save the result for access later. [Kerchunk](https://fsspec.github.io/kerchunk/) is a Python module that implements one way to store these catalogs, also known as virtual zarrs. 

In [ ]:
import warnings  # type: ignore
from collections.abc import Sequence

import earthaccess
import xarray as xr


def collapse_time_to_scalar(ds: xr.Dataset) -> xr.Dataset:
    """Collapse a SMAP L4 granule's single timestamp to a scalar coordinate.

    Each SMAP L4 granule carries one timestamp stored as a length-1 time
    variable along its own (non-indexed) dimension rather than as a scalar.
    When open_virtual_mfdataset concatenates with concat_dim="time",
    xarray tries to expand_dims("time") to create the stacking dimension
    and raises "time already exists as coordinate or variable name" because
    a non-scalar time variable is already present.

    Squeezing the length-1 dimension turns time into a scalar coordinate,
    which expand_dims is able to promote into a time dimension.

    Arguments:
        ds (xr.Dataset): A single (virtual) granule dataset.

    Returns:
        xr.Dataset: The dataset with time collapsed to a scalar coordinate.
    """
    if "time" in ds.variables:
        squeeze_dims = [d for d in ds["time"].dims if ds.sizes[d] == 1]
        if squeeze_dims:
            ds = ds.squeeze(squeeze_dims)
        ds = ds.set_coords("time")
    return ds


def virtualize_smap_l4(
    temporal: tuple[str, str],
    group: str = "Geophysical_Data",
    variables: Sequence[str] | None = ("sm_rootzone",),
    load: bool = False,
) -> xr.Dataset:
    """Build a virtual dataset of SMAP L4 geophysical variables over a time range.

    SMAP L4 (SPL4SMGP) stores the coordinate variables (time, x, y)
    in the file's root group, while the geophysical variables such as
    sm_rootzone live in the Geophysical_Data subgroup as 2-D (y, x)
    arrays with no time coordinate of their own.  Because
    open_virtual_mfdataset parses one group per call, the root group and the
    geophysical group are opened separately and the root coordinates are then
    attached to the stacked geophysical variables.

    Arguments:
        temporal (tuple[str, str]): (start, end) date strings passed to earthaccess.
        group (str): The subgroup holding the geophysical variables.
        variables (Sequence[str], optional): Variables to keep from group. None keeps the whole
            group; the default keeps only sm_rootzone.
        load (bool, optional): When False (default) the geophysical variables stay as
            VirtualiZarr ManifestArrays (byte-range references), which is
            what :func:`save_virtual_zarr` needs to write a compact kerchunk
            file.  When True they are materialised into a concrete,
            lazily-loaded dataset that can be computed on directly.

    Returns:
        A virtual xarray.Dataset (or a concrete one when load=True) with
        the requested variables stacked along a time dimension and carrying
        the time/x/y coordinates.
    """
    auth = earthaccess.login()
    if not auth.authenticated:
        auth.login(strategy="interactive", persist=True)

    warnings.filterwarnings("ignore", "As of version 1.0*", FutureWarning)
    results = earthaccess.search_data(
        short_name="SPL4SMGP",
        temporal=temporal,
    )

    open_options = {
        "access": "indirect",
        "concat_dim": "time",
        "coords": "minimal",
        "compat": "override",
        "combine_attrs": "override",
    }

    warnings.filterwarnings(
        "ignore",
        message="This DMRpp contains the variable EASE2_global_projection*",
        category=UserWarning
    )
    # The root group is always loaded so that time/x/y come back as concrete
    # coordinates; these are tiny and get inlined into the kerchunk file.
    result_root = earthaccess.virtualize(
        granules=results,
        load=True,
        data_vars="minimal",
        preprocess=collapse_time_to_scalar,
        loadable_variables=["time", "x", "y"],
        **open_options, # type: ignore
    )

    result_gph = earthaccess.virtualize(
        granules=results,
        load=load,
        group=group,
        data_vars="all",
        **open_options, # type: ignore
    )

    result = result_gph.assign_coords(
        time=result_root["time"],
        x=result_root["x"],
        y=result_root["y"],
    )

    if variables is not None:
        result = result[list(variables)]

    return result


The following cell will actually perform the virtualization operation, the time it takes depends on the length of the date range. For the date range used in this course (all of 2017), it will take about 2 hours to create the virtual zarr file. There will be no output from the code during this time, so please be patient. **This operation only needs to be performed once unless you want to virtualize a different time range of data.**

In [ ]:
import virtualizarr  # noqa: F401  (registers the .vz dataset accessor)

# Virtualize the full year of 2017, we will only use a subset of this data,
# but it doesn't hurt to have extra
date_range = ("2017-01-01", "2017-12-31")
vds_path = "inputs/SPL4SMGP_virtual_https.parquet"

ds = virtualize_smap_l4(date_range)
ds.vz.to_kerchunk(vds_path, format="parquet")

print(f"Saved virtual Zarr references to {vds_path}")